# Capstone: Mathematical Analysis of GPT

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 12.03 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/12_Capstone_Projects/04_mathematical_analysis_of_gpt.ipynb)

---

## 🎯 Learning Objective

Count parameters and FLOPs for a GPT-style model and derive its memory requirements, connecting the numbers to published scaling laws.

---

## 📐 Theory

`05.07` used the Chinchilla scaling law to answer "given a compute budget, how should I split it
between model size $N$ and data size $D$" — treating $N$ as a single number fed into a fitted
formula. This capstone opens up that number: where does $N$ actually come from, architecturally,
how many floating-point operations does using it cost, and how much memory does it actually
require to train or serve. These are the numbers that turn an architecture on paper into a
concrete hardware requirement.

### Counting parameters

A GPT-style transformer's parameters are dominated by the per-layer weight matrices. Each layer
has a self-attention block ($W_Q,W_K,W_V,W_O$, each $d_{\text{model}}\times d_{\text{model}}$ —
recall `01.10`) and a feed-forward block (two matrices, $d_{\text{model}}\times d_{ff}$ and
$d_{ff}\times d_{\text{model}}$, typically with $d_{ff}=4d_{\text{model}}$). This gives, per
layer, $4d_{\text{model}}^2 + 2d_{\text{model}}d_{ff} = 4d_{\text{model}}^2 +
8d_{\text{model}}^2 = 12\,d_{\text{model}}^2$ parameters (ignoring biases and LayerNorm, which
are a small correction). Across $n_{\text{layers}}$ layers, plus a token embedding table
($V\times d_{\text{model}}$, often **weight-tied** with the final output projection so it's
counted only once), this gives a closed-form parameter count — verified below against a real
PyTorch model's *actual* `numel()` count, not just the formula's own self-consistency.

### FLOPs: the "$2N$" and "$6N$" rules

For a single input token, a forward pass through one linear layer $W\in\mathbb{R}^{d_{in}\times
d_{out}}$ costs $2\,d_{in}d_{out}$ floating point operations (one multiply and one add per
weight, the standard FLOP-counting convention — recall matrix-vector products from `01.02`).
Summing this over every matmul in every layer gives total forward FLOPs-per-token that equals
**exactly** $2N$ (where $N$ is the matmul parameter count from above) — not an approximation,
verified below by explicit counting. Backpropagation (recall `02.10`) costs roughly twice the
forward pass's FLOPs (one pass to get gradients w.r.t. inputs, another w.r.t. weights), so a
full training step costs approximately $6N$ FLOPs per token — the exact quantity `05.07`'s
Chinchilla formula $C\approx 6ND$ uses, now derived rather than taken as given.

### Memory: training vs. inference

**Training memory** (using Adam, recall `04.04`/`12.03`) needs to store, per parameter: the
parameter itself, its gradient, Adam's first moment $m$, and second moment $v$ — four
same-sized buffers. At fp32 (4 bytes each), that's $16$ bytes per parameter, confirmed to match
published estimates for real models. **Inference memory** only needs the weights themselves
(often at fp16/bf16, 2 bytes each) — an order-of-magnitude smaller footprint, since there's no
gradient or optimizer state to keep. Inference has one more cost training doesn't:
the **KV cache** (recall `01.10`'s $K,V$ matrices) — every generated token requires storing that
token's keys and values for every layer and head, so they don't need recomputing for every
future token; this cache grows linearly with sequence length and batch size, and can become the
dominant memory cost for long-context, high-batch-size serving.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Breaking down the parameter-counting formula's three components (attention, feed-forward,
embedding) across the four real, published GPT-2 model sizes shows how the *embedding* table's
share shrinks as the model gets deeper and wider — it's fixed by vocabulary size, while the
per-layer matmul cost grows with $d_{\text{model}}^2$.

In [ ]:
# Real, published GPT-2 family configurations
gpt2_configs = {
    "GPT-2 small\n(124M)":  dict(n_layers=12, d_model=768,  vocab=50257),
    "GPT-2 medium\n(355M)": dict(n_layers=24, d_model=1024, vocab=50257),
    "GPT-2 large\n(774M)":  dict(n_layers=36, d_model=1280, vocab=50257),
    "GPT-2 XL\n(1.5B)":     dict(n_layers=48, d_model=1600, vocab=50257),
}

names, attn_list, ff_list, embed_list, total_list = [], [], [], [], []
for name, cfg in gpt2_configs.items():
    d_model, n_layers, vocab = cfg["d_model"], cfg["n_layers"], cfg["vocab"]
    d_ff = 4 * d_model
    attn = n_layers * 4 * d_model**2
    ff = n_layers * 2 * d_model * d_ff
    embed = vocab * d_model
    names.append(name)
    attn_list.append(attn / 1e6); ff_list.append(ff / 1e6); embed_list.append(embed / 1e6)
    total_list.append((attn + ff + embed) / 1e6)

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(names))
ax.bar(x, attn_list, label="attention ($4 n_L d_{model}^2$)", color="#4C72B0")
ax.bar(x, ff_list, bottom=attn_list, label="feed-forward ($8 n_L d_{model}^2$)", color="#DD8452")
bottom2 = np.array(attn_list) + np.array(ff_list)
ax.bar(x, embed_list, bottom=bottom2, label="embedding ($V \\cdot d_{model}$, tied)", color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("parameters (millions)")
ax.set_title("Formula-predicted parameter breakdown matches published\nGPT-2 model sizes (124M / 355M / 774M / 1.5B) closely")
ax.legend()
plt.tight_layout()
plt.show()

for name, total, embed in zip(names, total_list, embed_list):
    print(f"{name.split(chr(10))[0]}: formula total={total:.1f}M params, embedding share={100*embed/total:.1f}%")

## 🐍 Implementation from Scratch

We build an actual GPT-2-small-scale transformer in PyTorch and confirm the parameter-counting
formula matches its real `numel()` count almost exactly, verify the "$2N$ forward FLOPs" claim
by explicit per-operation counting (not approximation), and compute realistic training vs.
inference memory requirements including the KV cache.

In [ ]:
# --- Build an ACTUAL GPT-2-small-scale transformer and count its real parameters ---
class GPTBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn_qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.attn_out = nn.Linear(d_model, d_model, bias=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff1 = nn.Linear(d_model, d_ff, bias=True)
        self.ff2 = nn.Linear(d_ff, d_model, bias=True)

class GPTModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # standard GPT-2 weight tying

def count_params_unique(model):
    # counts each UNIQUE parameter tensor once, correctly handling the tied embedding/head
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p)); total += p.numel()
    return total

vocab_size, d_model, n_layers, d_ff, max_seq_len = 50257, 768, 12, 3072, 1024
model = GPTModel(vocab_size, d_model, n_layers, d_ff, max_seq_len)
actual_params = count_params_unique(model)

# Closed-form formula: 12 * n_layers * d_model^2 (attn+ff matmuls) + tied embedding + position embedding
N_matmuls = n_layers * (4 * d_model**2 + 2 * d_model * d_ff)
N_formula = N_matmuls + vocab_size * d_model + max_seq_len * d_model

print(f"GPT-2-small-scale model (d_model={d_model}, n_layers={n_layers}, vocab={vocab_size}):")
print(f"  Actual PyTorch param count (unique tensors, tied weights counted once): {actual_params:,}")
print(f"  Formula (12*n_layers*d_model^2 + tied embed + pos embed):               {N_formula:,}")
print(f"  Relative difference: {abs(N_formula - actual_params)/actual_params:.4f} "
      f"(remaining gap: biases + LayerNorm params, ignored by the simplified formula)")
print(f"  This matches OpenAI's published GPT-2-small size of ~124M parameters.")

# --- FLOPs: verify the "2N" forward-pass identity by EXPLICIT per-operation counting ---
def linear_flops(d_in, d_out):
    return 2 * d_in * d_out  # one multiply + one add per weight (standard FLOP convention)

per_layer_flops = (linear_flops(d_model, d_model) * 3    # Q, K, V projections
                    + linear_flops(d_model, d_model)      # output projection
                    + linear_flops(d_model, d_ff)          # ff1
                    + linear_flops(d_ff, d_model))         # ff2
total_forward_flops = n_layers * per_layer_flops

print(f"\nPer-token forward FLOPs, explicit sum over every matmul: {total_forward_flops:,}")
print(f"2 * N_matmuls (the standard '2N' approximation):          {2 * N_matmuls:,}")
print(f"Exact match (not an approximation, for the matmul-dominated part): "
      f"{total_forward_flops == 2 * N_matmuls}")
print(f"Full training step (forward + backward), the '6N' rule Chinchilla's C=6ND (05.07) uses:")
print(f"  6 * N_matmuls = {6 * N_matmuls:,} FLOPs per token")

# --- Memory: training (Adam) vs inference (weights + KV cache) ---
def bytes_for_dtype(dtype):
    return {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1}[dtype]

def training_memory_bytes(N, dtype="fp32"):
    # Adam needs 4 same-sized buffers: params, gradients, momentum m, variance v
    return N * 4 * bytes_for_dtype(dtype)

def inference_weight_memory_bytes(N, dtype="fp16"):
    return N * bytes_for_dtype(dtype)

def kv_cache_bytes(n_layers, n_heads, d_head, seq_len, batch_size, dtype="fp16"):
    return 2 * n_layers * n_heads * d_head * seq_len * batch_size * bytes_for_dtype(dtype)

N_7b = 7e9
n_heads_7b, d_model_7b, n_layers_7b = 32, 4096, 32
d_head_7b = d_model_7b // n_heads_7b

train_mem = training_memory_bytes(N_7b, "fp32")
inf_mem = inference_weight_memory_bytes(N_7b, "fp16")
print(f"\nAt a 7B-parameter scale (a common 'small' modern LLM size):")
print(f"  Training memory (Adam, fp32): {train_mem/1e9:.1f} GB "
      f"(params={N_7b*4/1e9:.1f} + grad={N_7b*4/1e9:.1f} + m={N_7b*4/1e9:.1f} + v={N_7b*4/1e9:.1f} GB)")
print(f"  Inference weight memory (fp16): {inf_mem/1e9:.1f} GB")
print(f"  Training-to-inference-weights ratio: {train_mem/inf_mem:.0f}x")

for seq_len, batch_size in [(2048, 1), (2048, 8), (8192, 1)]:
    kv_mem = kv_cache_bytes(n_layers_7b, n_heads_7b, d_head_7b, seq_len, batch_size, "fp16")
    print(f"  KV cache (seq_len={seq_len}, batch={batch_size}): {kv_mem/1e9:.2f} GB")

## 🤖 Why This Matters for AI

Every number derived above turns directly into a hardware purchasing or deployment decision.
Applying the memory formulas to three real model scales (7B, 13B, 70B — common sizes in the
current open-model ecosystem) against real GPU memory specifications (RTX 4090: 24GB, A100:
40GB/80GB, H100: 80GB) shows exactly why a 70B model needs multiple GPUs working together
(its weights alone exceed any single card's memory) while a 7B model comfortably fits on
consumer hardware even with a substantial KV cache. This is also exactly the calculation behind
a common real deployment question: "how many concurrent requests can this GPU serve?" — since
the KV cache (not the fixed weight memory) is what grows with batch size and context length,
there's a hard ceiling on how many sequences a given GPU can serve simultaneously once the
weights are loaded, computed directly below for a concrete case.

Concrete hardware context: an NVIDIA **A100** (80 GB) fits a 40B-parameter model in float16; an **H100** (80 GB) adds FP8 support, doubling effective throughput; a consumer **RTX 4090** (24 GB) fits ~12B parameters in float16 or ~25B in INT4. Systems like **Megatron-LM** and **DeepSpeed** implement tensor/pipeline parallelism to split models across multiple GPUs when they exceed a single card's memory.

---


In [ ]:
gpu_memory_gb = {"RTX 4090": 24, "A100 40GB": 40, "A100 80GB": 80, "H100": 80}

model_configs = {
    "7B":  dict(N=7e9,  n_layers=32, d_model=4096, n_heads=32),
    "13B": dict(N=13e9, n_layers=40, d_model=5120, n_heads=40),
    "70B": dict(N=70e9, n_layers=80, d_model=8192, n_heads=64),
}

print("Inference memory (weights + 4k-context KV cache, batch=1) vs. real GPU memory:")
results = {}
for model_name, cfg in model_configs.items():
    N = cfg["N"]
    weight_mem_gb = inference_weight_memory_bytes(N, "fp16") / 1e9
    d_head = cfg["d_model"] // cfg["n_heads"]
    kv_mem_gb = kv_cache_bytes(cfg["n_layers"], cfg["n_heads"], d_head, seq_len=4096, batch_size=1, dtype="fp16") / 1e9
    total_gb = weight_mem_gb + kv_mem_gb
    fits_on = [gpu for gpu, mem in gpu_memory_gb.items() if mem >= total_gb]
    results[model_name] = (weight_mem_gb, kv_mem_gb, total_gb, fits_on)
    print(f"  {model_name:>4}: weights={weight_mem_gb:6.1f}GB, KV cache={kv_mem_gb:5.2f}GB, "
          f"total={total_gb:6.1f}GB -- fits on: {fits_on if fits_on else '(none of these)'}")

# How many concurrent 4k-context requests can an 80GB GPU serve for the 7B model?
model_7b = model_configs["7B"]
weight_mem = inference_weight_memory_bytes(model_7b["N"], "fp16") / 1e9
available_for_kv = 80 - weight_mem
d_head_7b = model_7b["d_model"] // model_7b["n_heads"]
kv_per_seq = kv_cache_bytes(model_7b["n_layers"], model_7b["n_heads"], d_head_7b,
                             seq_len=4096, batch_size=1, dtype="fp16") / 1e9
max_batch = int(available_for_kv / kv_per_seq)
print(f"\n7B model on an 80GB GPU: {weight_mem:.1f}GB fixed for weights, "
      f"{available_for_kv:.1f}GB remaining for KV cache")
print(f"KV cache per sequence (4k context): {kv_per_seq:.3f}GB")
print(f"Maximum CONCURRENT 4k-context requests this GPU can serve: {max_batch}")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
names = list(model_configs.keys())
weights = [results[m][0] for m in names]
kv = [results[m][1] for m in names]
x = np.arange(len(names))
ax.bar(x, weights, label="weights (fp16)", color="#4C72B0")
ax.bar(x, kv, bottom=weights, label="KV cache (4k context, batch=1)", color="#C44E52")
for gpu, mem in gpu_memory_gb.items():
    ax.axhline(mem, ls=':', lw=1, color='gray')
    ax.text(len(names) - 0.4, mem + 1, gpu, fontsize=7, color='gray')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("memory (GB)")
ax.set_title("Model memory footprint vs. real GPU memory capacities\n(dotted lines mark actual GPU VRAM sizes)")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, using the formula $12\,n_{\text{layers}}\,d_{\text{model}}^2$, compute the
   (non-embedding) parameter count for a model with $n_{\text{layers}}=24$,
   $d_{\text{model}}=1024$ — GPT-2-medium's configuration. Verify against the Implementation
   section's `N_matmuls` calculation with these values substituted in, and check your total
   (plus embedding and position embedding) comes within 1% of the published 355M figure the
   Visual Intuition section uses — a numerical sanity check on the formula itself, not just
   self-consistency with the code.

<details>
<summary>💡 Solution</summary>

$12\times24\times1024^2 = 12\times24\times1{,}048{,}576 = 301{,}989{,}888 \approx 302$M — this
matches `N_matmuls` exactly, since `N_matmuls = n_layers*(4*d_model**2 + 2*d_model*d_ff)` with
$d_{ff}=4d_{\text{model}}$ simplifies to exactly $12\,n_{\text{layers}}\,d_{\text{model}}^2$
algebraically. Adding the tied embedding ($50257\times1024\approx51.5$M) and position embedding
($1024\times1024\approx1.0$M) gives a total of about $354.5$M, within $0.14\%$ of the published
$355$M figure — and within about $0.09\%$ of an actual constructed PyTorch model's real
`numel()` count, confirming the formula's small remaining gap really is just the ignored
biases and LayerNorm parameters, as the Implementation section states.

</details>

### 💭 UNDERSTAND
2. The Visual Intuition section shows the tied embedding table's share of total parameters
   shrinking as GPT-2 models get larger (deeper and wider). Without recomputing the real GPT-2
   configurations, explain precisely *why* this must happen from the formula alone: which term
   in $12\,n_{\text{layers}}\,d_{\text{model}}^2$ (matmuls) versus $V\cdot d_{\text{model}}$
   (embedding) grows faster as $d_{\text{model}}$ increases, and why vocabulary size $V$ being
   fixed (essentially a property of the tokenizer, not the model's capacity) is essential to the
   argument.

<details>
<summary>💡 Solution</summary>

The matmul term scales as $d_{\text{model}}^2$ while the embedding term scales only as
$d_{\text{model}}^1$ (linearly) — for a fixed vocabulary size $V$, increasing $d_{\text{model}}$
makes the matmul parameters grow quadratically while the embedding parameters grow only
linearly, so their *ratio* (embedding / total) must shrink toward $0$ as $d_{\text{model}}\to
\infty$, regardless of $n_{\text{layers}}$. This is essential precisely because $V$ doesn't grow
with the model: if vocabulary size scaled up alongside $d_{\text{model}}$ (say, proportionally),
the embedding term would also become quadratic in $d_{\text{model}}$ and the ratio would instead
approach some constant rather than shrinking to zero. In practice, GPT-2's tokenizer (and hence
$V\approx50257$) is fixed across all four model sizes in the family — only $d_{\text{model}}$
and $n_{\text{layers}}$ change — so this quadratic-vs-linear scaling gap is exactly what the
Visual Intuition section's bar chart is showing.

</details>

### ✏️ DERIVE
3. Derive a closed-form expression for the value of $d_{\text{model}}$ at which the matmul
   parameters ($12\,n_{\text{layers}}\,d_{\text{model}}^2$) exactly equal the (tied) embedding
   parameters ($V\cdot d_{\text{model}}$), as a function of $n_{\text{layers}}$ and $V$, and
   verify it numerically for a few values of $n_{\text{layers}}$ using GPT-2's vocabulary size.

<details>
<summary>💡 Solution outline</summary>

Set the two terms equal: $12\,n_{\text{layers}}\,d_{\text{model}}^2 = V\cdot d_{\text{model}}$.
Since $d_{\text{model}}>0$, divide both sides by $d_{\text{model}}$:
$$12\,n_{\text{layers}}\,d_{\text{model}} = V \;\;\Longrightarrow\;\; d_{\text{model}}^\star =
\frac{V}{12\,n_{\text{layers}}}.$$
At this crossover width, matmul and embedding parameters are exactly equal (each contributing
half the total, ignoring position embeddings and biases). For GPT-2's $V=50257$: at
$n_{\text{layers}}=12$, $d_{\text{model}}^\star\approx349$; at $n_{\text{layers}}=24$,
$d_{\text{model}}^\star\approx175$; at $n_{\text{layers}}=48$, $d_{\text{model}}^\star\approx87$.
Substituting each back in confirms the two terms match exactly (to floating-point precision) at
every one of these computed crossover widths — and since all four real GPT-2 configurations use
$d_{\text{model}}\ge768$, well above every one of these crossover values, the matmul term
already dominates for the entire real GPT-2 family, consistent with the Visual Intuition
section's embedding share dropping to single digits by GPT-2 XL.

</details>

### 🐍 IMPLEMENT
4. The Implementation section's `linear_flops`/`total_forward_flops` counts only the $Q,K,V,O$
   projection matmuls and the feed-forward matmuls — it doesn't count $QK^\top$ or
   (attention-weights)$\times V$, whose cost scales with *sequence length*, not just
   $d_{\text{model}}$. Add per-layer FLOPs for both of these (each costing roughly
   $2\cdot\text{seq\_len}\cdot d_{\text{model}}$ per token) to the total, and sweep sequence
   length to find where this previously-ignored cost becomes comparable to the projection
   matmuls' cost, for GPT-2-small's $d_{\text{model}}=768$.

<details>
<summary>✅ How to know you're right</summary>

At `seq_len=2048`, the added attention-score FLOPs should come out to a substantial fraction of
the original projection-only total (in one computation, about 44% on top of the
projection-only total for GPT-2-small's configuration) — not a negligible correction, confirming
why real FLOP accounting for long-context models can't just use the "$2N$" projection-only
shortcut. Setting the per-layer attention-score cost ($4\cdot\text{seq\_len}\cdot d_{\text{model}}$,
combining both $QK^\top$ and $AV$) equal to the per-layer projection cost and solving for
`seq_len` should give a crossover point (algebraically,
$\text{seq\_len}^\star = \text{per\_layer\_proj\_flops}/(4\,d_{\text{model}})$); for GPT-2-small
this comes out to `seq_len` in the thousands — you should find that GPT-2's actual training
context length (1024) sits below this crossover, meaning the ignored cost, while non-negligible,
was not yet dominant at GPT-2's own context length, but would become so at several times that
length.

</details>

### 🤖 APPLY
5. LLaMA-7B uses a genuinely different architecture from GPT-2: it does **not** tie the
   embedding and output-head matrices (both are counted separately), and its feed-forward block
   is a SwiGLU (3 weight matrices — `W1`, `W2`, `W3` — instead of GPT-2's 2), with hidden
   dimension computed as `d_ff = round_up_to_multiple_of_256(int(2/3 * 4*d_model))` rather than
   simply `4*d_model`. Using LLaMA-7B's published configuration
   ($n_{\text{layers}}=32$, $d_{\text{model}}=4096$, $\text{vocab}=32000$), adapt this
   notebook's parameter-counting approach to LLaMA's architecture and check your total against
   the model's published 6.7B parameter count.

<details>
<summary>✅ What you should observe</summary>

Correctly adapting the formula (SwiGLU's 3 matrices instead of 2, the 2/3-scaled and
256-rounded hidden dimension instead of a flat $4d_{\text{model}}$, and an *untied* output head
counted as its own $V\times d_{\text{model}}$ term) should land within about half a percent of
the published 6.7B figure. If you instead naively reuse GPT-2's formula unchanged (2 FFN
matrices at $d_{ff}=4d_{\text{model}}$, tied embedding), you'll get a total that's noticeably
off (in one computation, about a 1.9% relative error versus 0.6% for the correctly-adapted
formula) — small in isolation, but the point of this exercise is that "count the parameters"
requires knowing the *specific* architecture's actual matrix shapes and tying choices, not
applying one memorized formula unconditionally to every transformer variant.

</details>

### 🚀 CHALLENGE
6. Real inference serving often uses **quantization** (e.g. int8 or int4 weights) to reduce
   memory further. Add an `"int4"` option to `bytes_for_dtype` (0.5 bytes per parameter) and
   recompute the 70B model's total inference memory (weights + 4k-context KV cache) at int4
   weights — does it now fit on a single 80GB GPU? Then investigate: does quantizing the KV
   cache itself (not just the weights) to int8 or int4 change the picture much at this scale,
   and why might KV cache quantization be a riskier decision, accuracy-wise, than weight
   quantization?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer shows the arithmetic clearly: at fp16 weights, the 70B model's weights alone
take about 140GB, exceeding even an 80GB card before the KV cache is added at all; at int8,
about 70GB (still tight, roughly the KV cache's few GB pushes it over 80GB combined); at int4,
weights drop to about 35GB, and combined with a fp16 KV cache (roughly 10-11GB at 4k context,
batch 1) the total comes in around 45-46GB — comfortably fitting on an 80GB card, with room to
spare for a larger batch or context than batch=1 alone would need. Quantizing the KV cache too
(to int8 or int4) shaves off further GB, but the *absolute* savings are much smaller than
quantizing the weights, since at this scale the KV cache is already a small fraction of total
memory next to a 70B-parameter model's weights — the leverage is in the weights first. On the
accuracy-risk question, a strong answer reasons (or looks up) that weight quantization is a
one-time, offline decision applied to values that don't change during inference and that can be
calibrated carefully in advance, whereas the KV cache stores *activations* computed freshly for
every new input — these have a much less predictable, potentially wider dynamic range across
different prompts and positions, so aggressively quantizing them risks compounding error into
every subsequent token's attention computation in a way that's harder to calibrate for once,
offline, before serving ever begins.

</details>

---

## 📚 Further Reading
- Kaplan et al., [\"Scaling Laws for Neural Language Models\"](https://arxiv.org/abs/2001.08361)
- Narayanan et al., [\"Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM\"](https://arxiv.org/abs/2104.04473)
- Pope et al., [\"Efficiently Scaling Transformer Inference\"](https://arxiv.org/abs/2211.05102)

---

*Next notebook: [Capstone: Read a Research Paper Mathematically](05_read_a_research_paper.ipynb)*